# gsd — the SPTLS reading suite (diagnostics extension)

A companion to `gsd_quickstart.ipynb`. Where the quickstart *fits and
forecasts*, this notebook *reads* the fitted operator — the SPTLS analogue of
what ACF/PACF and the unit-circle root plot are to Box–Jenkins.

> **These are descriptive readings of the fitted operator, not statistical
> tests.** Formal tests built on them (for stationarity, shared directions,
> directional coupling) are under development and will arrive with their null
> distributions. For now, read the numbers as indicative geometry.

`sp.report()` returns a `SPTLSReport` with `.summary()` (the numbers) and
`.plot()` (a visual companion — mostly pedagogical; practitioners usually read
the table).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../../lib"))
import numpy as np, gsd
np.set_printoptions(precision=3, suppress=True)

## 1. Univariate reading

Fit SPTLS on one series and print the report. The blocks:

- **regime (descriptive)** — a label from the operator's spectral radius and
  rotor angle: stable / near-unit / explosive, oscillatory or not.
- **rotor angle / stretch gains / antisymmetry index** — the polar read of the
  operator: how much it rotates per step vs how much it stretches/damps.
- **operator spectrum** — the eigenvalues (the SPTLS "roots"): a complex pair's
  modulus is the per-step damping, its argument the oscillation frequency.
- **grade energy** — how the dynamics split across scalar / stretch / rotation.

In [ ]:
y = gsd.datasets.rotation(T=200)
rep = gsd.SPTLS().fit(y).report()
print(rep.summary())

### The visual companion

`.plot()` shows, for a univariate fit: the operator spectrum in the complex
plane (with the unit circle), the grade-energy bars, and the PGCF grade-0 vs
grade-2 curves. The grade-2 curve being non-zero is the signature of
rotational content a power spectrum cannot carry.

In [ ]:
rep.plot();   # remove the semicolon's effect by just calling it in Jupyter


### A contrasting process

The same reading on a near-random-walk: the regime reads *near-unit*, the
rotor angle is small, and the grade energy concentrates away from rotation.

In [ ]:
rw = gsd.datasets.random_walk(T=300, seed=1)
print(gsd.SPTLS().fit(rw).report().summary())

## 2. Multivariate reading

For several series the report adds the **cross structure**:

- **directed influence** `||block(i<-j)||` — how much series *j*'s jet moves
  series *i* (off-diagonal blocks of the stacked operator), and its
  **rotational** (phase-coupled) part.
- **source-channel effect** — whether it is the *level*, *velocity* or
  *acceleration* of *j* that moves the level of *i*.
- **shared-direction reading** — near-null directions of the joint embedding
  Gram, read as candidate near-stationary combinations of the levels (a
  descriptive companion to a cointegration test, not a test in itself).

In [ ]:
Y = gsd.datasets.cointegrated_var(T=400)   # two cointegrated I(1) series
mrep = gsd.SPTLS().fit(Y).report()
print(mrep.summary())

In [ ]:
mrep.plot(series=0);   # spectrum (y0) + influence heatmap + joint-Gram eigenvalues


### Reading the cross structure

For the synthetic panel above, built with the cointegrating combination
$y_0 - y_1$, the shared-direction reading recovers a most-stationary level
combination close to $(+0.707)\,y_0 + (-0.707)\,y_1$ — i.e. $y_0-y_1$ up to
scale — and the directed-influence matrix shows which series moves which,
split into coaxial and rotational parts.

In [ ]:
print("most-stationary level combination:", np.round(mrep.cointegrating_vector, 3))
print("near-stationary directions (count):", mrep.coint_rank)
print("directed influence ||i<-j||:\n", np.round(mrep.influence, 3))

## Caveats and what's next

- **Not tests.** Every number above describes the *fitted operator*; none is a
  hypothesis test with a null distribution. Formal tests are in development.
- The regime label uses the *linear* operator, so on strongly nonlinear /
  chaotic series it is uninformative — there a grade-2 (quadratic) dictionary
  is needed (see the univariate OLS-vs-SPTLS walkthrough).
- A directed **causal graph** over the influence matrix is a planned next
  iteration; for now the influence matrix carries the same information.